# 04 – Phase 1 Report: Backtest Results & Regime Analysis

This notebook loads the Phase 1 artefacts and visualises the key results.

## Setup

In [ ]:
import sysfrom pathlib import Pathsys.path.insert(0, str(Path.cwd()))import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom loguru import loggerfrom src.analysis import regime_attribution as rafrom src.visualization import phase1_charts as chartsOUT_DIR = Path("data/_meta/phase1")pd.set_option("display.float_format", "{:.4f}".format)

## 1. Backtest Metrics

In [ ]:
metrics = pd.read_csv(OUT_DIR / "backtest_metrics.csv", index_col=0)metrics.style.background_gradient(cmap="RdYlGn", subset=["ann_return", "sharpe", "calmar"])

## 2. Equity Curves

In [ ]:
equity = pd.read_csv(OUT_DIR / "equity_curves.csv", index_col=0, parse_dates=True)fig = charts.equity_curve_plot(equity, title="Strategy Equity Curves (log scale)")fig.show()# Also saved as PNG:# equity_curves.png

## 3. Drawdown Curves

In [ ]:
fig = charts.drawdown_plot(equity, title="Strategy Drawdowns")fig.show()

## 4. Regime Distribution

In [ ]:
regime_labels = pd.read_csv(OUT_DIR / "regime_labels.csv", index_col=0, parse_dates=True)counts = regime_labels["regime"].value_counts()ax = counts.plot(kind="bar", color=["#2ecc71", "#e74c3c", "#f39c12", "#3498db", "#95a5a6"])ax.set_title("Regime Distribution")ax.set_ylabel("Months")plt.tight_layout()plt.show()counts.to_frame("months")

## 5. Regime Heatmap – Mean Monthly Returns

In [ ]:
panel = pd.read_csv(OUT_DIR / "monthly_panel.csv", index_col=0, parse_dates=True)panel["regime"] = regime_labels["regime"]heatmap = ra.regime_heatmap(panel)fig = charts.regime_heatmap_plot(heatmap)fig.show()heatmap

## 6. Event Window Analysis

In [ ]:
event_rets = ra.event_window_returns(panel)fig = charts.event_window_bar(event_rets)fig.show()event_rets

## 7. Calendar-Year S&P 500 Returns

In [ ]:
annual = ra.calendar_year_returns(panel, "sp500_ret")fig = charts.annual_return_bar(annual)fig.show()annual.to_frame("sp500_return")

## 8. Regime × Asset Summary Table

In [ ]:
attr_table = ra.regime_asset_table(panel)attr_table

## 9. Key Findings

### Counter-Intuitive Results1. **"Recession" months show the highest equity returns** (mean +1.55%/mo for S&P 500).   This is because INDPRO YoY (our growth proxy) is a *coincident* indicator — growth has already   turned negative when the regime is labelled, but markets often bottom before the economy.2. **10Y Treasury yield "returns" are unreliable** — using pct_change of yield as a bond   price proxy produces nonsensical results during rapid rate changes (e.g. +142% in 2022).   A bond ETF (IEF/TLT) should replace this in Phase 2.3. **Regime rotation improves risk-adjusted returns but not absolute returns** — Sharpe 0.64   vs S&P 500 0.67, but lower CAGR (7.7% vs 9.7%) and lower MDD (-30.3% vs -47.5%).### Known Data Issues- **us_pmi (NAPM)**: FRED CSV endpoint returns 404. Using INDPRO YoY as fallback.- **china_cpi**: akshare API format changed — date parsing fails.- **ea_hicp**: ECB SDW SSL handshake fails intermittently.- **csi300**: yfinance ticker 000300.SH appears delisted/renamed.

## 10. Sensitivity Analysis

In [ ]:
SENS_DIR = Path("data/_meta/sensitivity")sensitivity = pd.read_csv(SENS_DIR / "sensitivity_metrics.csv")best = sensitivity.loc[sensitivity["ann_return"].idxmax()]print(f"Best params: growth_threshold={best["growth_threshold"]}, cpi_high={best["cpi_high"]}")print(f"Best CAGR: {best["ann_return"]:.4f}")pivot = sensitivity.pivot_table(index="growth_threshold", columns="cpi_high", values="ann_return")pivot.style.background_gradient(cmap="RdYlGn")